# OpenAI Agents SDK on Fabric

Runs unmodified OpenAI Agents SDK workloads against a Fabric inference endpoint: function tools, handoffs, structured output, streaming, and a Fabric run trace with a token and step budget.

You need a Fabric API key (console → API keys). The key is exchanged for a short-lived inference token; the key itself never reaches the gateway. Tracing is disabled so nothing is sent to OpenAI.

In [ ]:
!pip install -q openai-agents openai httpx pydantic


In [ ]:
import json, time, uuid
from getpass import getpass
import httpx
from openai import AsyncOpenAI
from pydantic import BaseModel
from agents import (Agent, ModelSettings, Runner, function_tool, set_default_openai_api,
                    set_default_openai_client, set_tracing_disabled)

CONTROL_URL = "https://api.nscale.host"
INFERENCE_URL = "https://inference.nscale.host/v1"
MODEL = "qwen3.5-2b"          # the API model name of your deployment
API_KEY = getpass("Fabric API key: ")


## Connect the SDK to Fabric
Exchange the API key for an inference token (valid for a limited time; rerun this cell to refresh it), then point the SDK at the Fabric gateway. Fabric serves the Chat Completions API, so the SDK is switched to it.

In [ ]:
def get_token() -> str:
    r = httpx.post(f"{CONTROL_URL}/v1/token", timeout=20, headers={"User-Agent": "fabric-colab"},
                   json={"grant_type": "api_key", "api_key": API_KEY, "audience": "fabric-inference"})
    r.raise_for_status()
    return r.json()["access_token"]

TOKEN = get_token()
client = AsyncOpenAI(base_url=INFERENCE_URL, api_key=TOKEN, timeout=180, max_retries=1)
set_default_openai_client(client, use_for_tracing=False)
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

# Every model call carries one Fabric run id with a token and step budget, so the gateway
# can enforce the budget and record a per-step trace (see the last cell).
RUN_ID = f"colab-{uuid.uuid4().hex[:8]}"
settings = ModelSettings(
    temperature=0.2,
    max_tokens=512,                      # keep replies bounded on small GPUs
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},   # Qwen3.5: answer directly
    extra_headers={"X-Fabric-Run-Id": RUN_ID,
                   "X-Fabric-Run-Budget-Tokens": "40000",
                   "X-Fabric-Run-Budget-Steps": "20"},
)
print("run id:", RUN_ID)


## Tools (local Python functions the model can call)

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Current weather for a city."""
    data = {"lisbon": "18°C, sunny", "london": "9°C, rain", "paris": "12°C, cloudy"}
    return data.get(city.lower(), "15°C, clear")


@function_tool
def search_flights(origin: str, destination: str) -> str:
    """Flights between two cities with prices in EUR."""
    return json.dumps([{"flight": "TP1351", "price": 89}, {"flight": "BA500", "price": 142}])


@function_tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies."""
    rates = {("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93, ("EUR", "GBP"): 0.85}
    return f"{amount * rates.get((from_currency.upper(), to_currency.upper()), 1.0):.2f} {to_currency.upper()}"


## 1. An agent with function tools

In [ ]:
agent = Agent(name="Travel assistant", model=MODEL, model_settings=settings,
              instructions="Use the tools to answer. Be brief.",
              tools=[get_weather, search_flights, convert_currency])
t = time.time()
result = await Runner.run(agent, "What's the weather in Lisbon, and what's the cheapest flight from London to Lisbon in USD?")
print(f"{time.time()-t:.1f}s\n{result.final_output}")


## 2. Handoffs: a triage agent routes to a specialist

In [ ]:
weather_agent = Agent(name="Weather agent", model=MODEL, model_settings=settings,
                      handoff_description="Answers weather questions", tools=[get_weather],
                      instructions="Answer weather questions with the tool. Be brief.")
flights_agent = Agent(name="Flights agent", model=MODEL, model_settings=settings,
                      handoff_description="Finds flights", tools=[search_flights],
                      instructions="Find flights with the tool. Be brief.")
triage = Agent(name="Triage", model=MODEL, model_settings=settings,
               handoffs=[weather_agent, flights_agent],
               instructions="Hand off to the right specialist. Do not answer yourself.")
t = time.time()
result = await Runner.run(triage, "Is it raining in London right now?")
print(f"{time.time()-t:.1f}s  answered by: {result.last_agent.name}\n{result.final_output}")


## 3. Structured output (Pydantic)
Small models can loop when a JSON schema and tools share one request, so one agent gathers facts with tools and a second, tool-free agent fills the schema. Larger models can do both in one agent.

In [ ]:
class Trip(BaseModel):
    destination: str
    weather: str
    flight: str
    price_eur: int
    packing: list[str]

researcher = Agent(name="Researcher", model=MODEL, model_settings=settings,
                   tools=[get_weather, search_flights],
                   instructions="Look up the weather at the destination and the flights. Report the facts briefly.")
formatter = Agent(name="Formatter", model=MODEL, model_settings=settings, output_type=Trip,
                  instructions="Turn the research notes into the trip plan. Pick the cheapest flight. Suggest 3-5 packing items.")
t = time.time()
notes = await Runner.run(researcher, "Plan a trip from London to Lisbon.")
plan = await Runner.run(formatter, notes.final_output)
print(f"{time.time()-t:.1f}s")
plan.final_output


## 4. Streaming, with tool calls as they happen

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent

streamer = Agent(name="Streamer", model=MODEL, model_settings=settings, tools=[get_weather],
                 instructions="Use tools when useful. Be brief.")
run = Runner.run_streamed(streamer, "Weather in Paris, then one sentence on what to wear.")
async for event in run.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
    elif event.type == "run_item_stream_event" and event.item.type == "tool_call_item":
        print(f"\n  -> tool call: {event.item.raw_item.name}", flush=True)


## 5. The Fabric run trace
Every call above ran under one run id. The gateway reports tokens used, steps, budget remaining, and each step's latency and tool calls, including any malformed tool arguments it repaired.

In [ ]:
r = httpx.get(f"{INFERENCE_URL}/runs/{RUN_ID}", headers={"Authorization": f"Bearer {TOKEN}"}, timeout=20)
r.raise_for_status()
trace = r.json()
print(json.dumps(trace["totals"], indent=2))
for step in trace.get("steps", []):
    calls = ", ".join(c["name"] for c in step.get("tool_calls", [])) or "-"
    print(f"step {step['index']:>2}  {step.get('latency_ms', 0):>6} ms  tools: {calls}")
